# Dependency Parsing: The Network Engineer

**Project Brief**

Welcome, **Network Engineer**.

Previously (in Constituency Parsing), we built buildings. Now, we are wiring a network.
In **Dependency Grammar**, words are nodes. We connect them with directed links (dependencies) to form a specialized network (tree).

Your Job:
1.  **Analyze the Nodes**: Identify Heads and Dependents.
2.  **Operate the Switchboard**: Use a **Transition-Based Parser** (Shift-Reduce) to build connections.
3.  **Optimize the Grid**: Understand Graph-Based approaches.
4.  **Network Diagnostics**: Evaluate parsing accuracy using UAS and LAS.

---

In [1]:
# Network Tools (Setup)
!pip install spacy nltk pandas numpy
!python -m spacy download en_core_web_sm

import spacy
from spacy import displacy
import pandas as pd
import nltk
from collections import deque


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 1. Dependency Relations (The Circuit)

Every connection is binary and asymmetric:
*   **Head (Governor)**: The central processing unit (e.g., the Verb in a sentence).
*   **Dependent (Modifier)**: A node attached to the Head.

**Universal Dependencies (UD)** Tags:
*   `nsubj` (Nominal Subject): "**Cars** drive."
*   `dobj` (Direct Object): "Drive **cars**."
*   `det` (Determiner): "**The** car."

In [2]:
# Visualizing the Network
nlp = spacy.load("en_core_web_sm")
doc = nlp("The autonomous car drove gracefully")

print("{:<15} {:<10} {:<10} {:<20}".format('Token', 'Tag', 'Dep', 'Head'))
print("-" * 55)
for token in doc:
    print("{:<15} {:<10} {:<10} {:<20}".format(
        token.text, token.pos_, token.dep_, token.head.text
    ))

# displacy.render(doc, style='dep', jupyter=True) # View the wiring diagram

Token           Tag        Dep        Head                
-------------------------------------------------------
The             DET        det        car                 
autonomous      ADJ        amod       car                 
car             NOUN       nsubj      drove               
drove           VERB       ROOT       drove               
gracefully      ADV        advmod     drove               


## 2. Transition-Based Parsing (The Switchboard)

How does a computer build this tree in one pass?
We use a **Shift-Reduce Parser** (Arc-Standard System).

**The Machine State**:
1.  **Stack ($\sigma$)**: Active nodes we are processing.
2.  **Buffer ($\beta$)**: Incoming words (queue).
3.  **Arcs ($A$)**: The connections we've built.

**The Controls (Transitions)**:
1.  **SHIFT**: Move word from Buffer to Stack.
2.  **LEFT-ARC**: Head is on Stack-top, Dependent is below it. Draw arrow $Stack_1 \rightarrow Stack_0$. Pop $Stack_1$ (Stack-0 remains as head).
3.  **RIGHT-ARC**: Head is below Stack-top, Dependent is on Stack-top. Draw arrow $Stack_1 \rightarrow Stack_0$. Pop $Stack_0$.

## 3. Implementation: Building the Router

### Manual Trace (Dry Run)
Sentence: "He sent it"
Goal: `sent -> He (nsubj)`, `sent -> it (dobj)`, `ROOT -> sent`

| Step | Stack | Buffer | Action | Result |
|---|---|---|---|---|
| 0 | [ROOT] | [He, sent, it] | Start | |
| 1 | [ROOT, He] | [sent, it] | SHIFT | |
| 2 | [ROOT, He, sent] | [it] | SHIFT | |
| 3 | [ROOT, sent] | [it] | LEFT-ARC (nsubj) | arc(sent, He) added. 'He' popped. |
| 4 | [ROOT, sent, it] | [] | SHIFT | |
| 5 | [ROOT, sent] | [] | RIGHT-ARC (dobj) | arc(sent, it) added. 'it' popped. |
| 6 | [ROOT] | [] | RIGHT-ARC (root) | arc(ROOT, sent) added. |

Now, let's code this machine.

In [3]:
class ShiftReduceParser:
    def __init__(self):
        self.stack = [] # The Stack [Root, ...]
        self.buffer = [] # The Buffer [w1, w2, ...]
        self.arcs = [] # The connections [(Head, Label, Dep)]

    def parse(self, sentence, oracle_moves):
        # Initialize
        self.stack = ['ROOT']
        self.buffer = list(sentence.split())
        self.arcs = []
        
        print(f"Initial State: Stack: {self.stack} | Buffer: {self.buffer}")
        
        # Execute moves
        for move in oracle_moves:
            if move == 'SHIFT':
                if not self.buffer:
                    print("Error: Cannot shift, buffer empty")
                    break
                word = self.buffer.pop(0)
                self.stack.append(word)
                
            elif move.startswith('LEFT-ARC'):
                # Arc-Standard Left-Arc: 
                # top=w2 (Head), second=w1 (Dep). Arc: w2 -> w1. Remove w1.
                label = move.split(':')[1] if ':' in move else 'dep'
                if len(self.stack) < 2:
                    print("Error: Stack too small for Left-Arc")
                    break
                    
                head = self.stack[-1] # Top
                dep = self.stack.pop(-2) # Second item
                self.arcs.append((head, label, dep))
                
            elif move.startswith('RIGHT-ARC'):
                # Arc-Standard Right-Arc:
                # top=w2 (Dep), second=w1 (Head). Arc: w1 -> w2. Remove w2.
                label = move.split(':')[1] if ':' in move else 'dep'
                if len(self.stack) < 2:
                    print("Error: Stack too small for Right-Arc")
                    break
                    
                head = self.stack[-2] # Second item
                dep = self.stack.pop(-1) # Top item
                self.arcs.append((head, label, dep))
            
            print(f"Move: {move:<15} | Stack: {str(self.stack):<30} | Arcs: {self.arcs}")
            
        return self.arcs

# Manual Test Run
parser = ShiftReduceParser()
sentence = "He sent it"
# The Oracle sequence we derived manually:
moves = ['SHIFT', 'SHIFT', 'LEFT-ARC:nsubj', 'SHIFT', 'RIGHT-ARC:dobj', 'RIGHT-ARC:root']

final_arcs = parser.parse(sentence, moves)
print("\nFinal Connections:", final_arcs)

Initial State: Stack: ['ROOT'] | Buffer: ['He', 'sent', 'it']
Move: SHIFT           | Stack: ['ROOT', 'He']                 | Arcs: []
Move: SHIFT           | Stack: ['ROOT', 'He', 'sent']         | Arcs: []
Move: LEFT-ARC:nsubj  | Stack: ['ROOT', 'sent']               | Arcs: [('sent', 'nsubj', 'He')]
Move: SHIFT           | Stack: ['ROOT', 'sent', 'it']         | Arcs: [('sent', 'nsubj', 'He')]
Move: RIGHT-ARC:dobj  | Stack: ['ROOT', 'sent']               | Arcs: [('sent', 'nsubj', 'He'), ('sent', 'dobj', 'it')]
Move: RIGHT-ARC:root  | Stack: ['ROOT']                       | Arcs: [('sent', 'nsubj', 'He'), ('sent', 'dobj', 'it'), ('ROOT', 'root', 'sent')]

Final Connections: [('sent', 'nsubj', 'He'), ('sent', 'dobj', 'it'), ('ROOT', 'root', 'sent')]


## 4. Projectivity (The Crossing Wires Problem)

Our Shift-Reduce parser has a fatal flaw: it can only handle **Projective** trees.
A tree is projective if you can draw all arcs above the sentence without any lines crossing.

**Example of Non-Projective**:
"A hearing is scheduled on the issue today."
*   `scheduled` -> `today` (Adverb modifier)
*   `scheduled` -> `on the issue` (PP modifier)
*   BUT `hearing` is between `scheduled` and `today`.
*   If `hearing` connects to something outside, we might cross lines.

English is mostly projective. Languages like Czech or German have many non-projective structures (free word order). For those, we need **Graph-Based Parsing** or a specialized "Swap" transition.

## 5. Graph-Based Parsing (The Mesh Network)

Transition-based is greedy (fast, but makes mistakes). 
**Graph-Based** parsing considers **every possible edge** between every two words in the sentence.

It creates a complete directed graph where edge weights are scores (probability of dependency).
Then, we find the **Maximum Spanning Tree (MST)**.

**Chu-Liu-Edmonds Algorithm**:
1.  Greedily select best incoming edge for every node.
2.  If tree, done.
3.  If cycle, contract cycle into super-node and repeat.

## 6. Evaluation (Diagnostics)

We compare our generated arcs to the Gold Standard.

1.  **UAS (Unlabeled Attachment Score)**: 
    $\frac{\text{# Correct Heads}}{\text{Total Words}}$
2.  **LAS (Labeled Attachment Score)**: 
    $\frac{\text{# Correct Heads AND Correct Labels}}{\text{Total Words}}$

LAS is always <= UAS.

**Project Completed, Engineer.** The data packets are flowing correctly.